# 02 — DGCA Website Monitor (Phase 2)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 2 of 8 — "What's New" Scraper

## 1. Project Overview

Phase 1 built and tested the Outlook email pipeline in isolation (dry-run,
`TEST_MODE=true`). This notebook builds the second independent piece: a
scraper that extracts structured publication metadata from DGCA's
**"What's New"** section at https://www.dgca.gov.in/digigov-portal/ —
**title, category, published date, publication URL, PDF URL** — for each of
its four categories: **Tenders, Regulation, Guidance, Announcements**.

**Scope of this phase:** read-only extraction of current "What's New" listings
into an in-memory structure and a timestamped JSON snapshot under
`data/raw/`. Deliberately out of scope here (later phases): PDF downloading,
PostgreSQL change-detection storage, old-vs-new comparison, AI summarization,
email integration, the dashboard.


## 2. Website Inspection Findings

Per the project's own rule, the live site was inspected in a browser
(network tab + JS source) before writing any scraper code, rather than
guessing at HTML structure. Findings:

**The "What's New" widget mixes two different data sources:**

1. **Regulation, Guidance, Announcements** are loaded by client-side JavaScript
   (`ajaxCallService(classification)`) that calls a genuine **JSON API** —
   no HTML scraping needed for these three:

   ```
   POST https://www.dgca.gov.in/digigov-portal/scan
        ?baseLocale=en_US&requestType=ApplicationRH
        &actionVal=viewAllServiceForAJAX&screenId=10000001
        &classification={Regulation|Guidance|Announcement}
   ```

   Response JSON shape (parallel arrays, one entry per publication):
   ```json
   {
     "totalRecords": 3,
     "contents": ["<full publication text, doubles as the title>", "..."],
     "startDate": ["03/08/2026", "..."],
     "attachId": ["CLtgzthDuOwIBNZ6uyXHXg%3D%3D", "..."]
   }
   ```

2. **Tenders** has no such AJAX call (its tab link has no `onclick` handler) —
   it is rendered server-side into the homepage HTML at `<div id="tenders">`,
   inside repeated `.new-window-pdf` blocks. It also defaults to **Hindi**
   content on a fresh request; matching the site's own client-side behavior,
   English is obtained by first POSTing a locale-change request in the same
   session, then re-fetching the homepage:

   ```
   POST .../scan?baseLocale=en_US&requestType=ApplicationRH&actionVal=localeChange&screenId=10000001
   GET  https://www.dgca.gov.in/digigov-portal/          (now renders English)
   ```

3. **PDF links** for every category resolve through the same JS helper,
   confirmed by reading `pdfDownload()`'s source in the page:
   ```
   https://www.dgca.gov.in/digigov-portal/Upload?flag=iframeAttachView&attachId={attachId}&baseLocale=en_US
   ```
   There is no separate "detail page" per publication — the title link *is*
   the PDF link. `publication_url` is therefore recorded as the DGCA
   "What's New" portal page itself (the only place to browse the listing),
   and `pdf_url` as the resolved attachment link.

4. **Encoding gotcha:** the API response has no `Content-Type` charset, so
   `requests` misdetects it. Special characters (smart quotes, en-dashes) are
   valid UTF-8 on the wire but come out as mojibake unless
   `response.encoding` is explicitly forced to `"utf-8"` before `.json()`.

**Conclusion: no browser automation (Playwright) is needed.** A plain
`requests.Session` — homepage GET, optional locale-change POST, then the
JSON/HTML fetches above — reproduces everything the browser does. Playwright
is deliberately not installed for this phase; it can be revisited later if
DGCA changes the site to be more JS-dependent.


## 3. Imports

In [ ]:
import json
import logging
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional
from urllib.parse import unquote

import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel

print("Imports OK")


## 4. Configuration

No secrets are needed for this phase (public website). Configuration here is
just endpoint/timeout constants -- kept as plain module-level values rather
than environment variables, since none of them are sensitive or
deployment-specific.


In [ ]:
BASE_URL = "https://www.dgca.gov.in/digigov-portal/"
SCAN_URL = BASE_URL + "scan"
UPLOAD_URL = BASE_URL + "Upload"

REQUEST_TIMEOUT_SECONDS = 20
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120 Safari/537.36"
)

# AJAX-backed categories (JSON API) vs HTML-only categories.
AJAX_CATEGORIES = ["Regulation", "Guidance", "Announcement"]
HTML_CATEGORIES = ["Tender"]
ALL_CATEGORIES = AJAX_CATEGORIES + HTML_CATEGORIES

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOG_DIR = PROJECT_ROOT / "logs"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOG_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_URL       : {BASE_URL}")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"Categories     : {ALL_CATEGORIES}")


## 5. Logging

Same pattern as the Phase 1 notebook: console + a per-day log file under
`logs/`. Configured early (unlike Phase 1, where it was configured later)
because the scraper functions log as they run, and we want that visible from
the first real fetch onward.


In [ ]:
LOG_FILE = LOG_DIR / f"scraper_{datetime.now(timezone.utc).strftime('%Y%m%d')}.log"

logger = logging.getLogger("dgca_scraper")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.propagate = False

logger.info("Scraper logger initialized.")
logger.info(f"Log file: {LOG_FILE}")


## 6. Data Model

`DGCAPublication` is the scraper's output contract -- a raw, not-yet-analyzed
publication record. It is intentionally simpler than Phase 1's `DGCAUpdate`
(no summary/impact fields yet -- those are produced by later phases from this
data). `source_id` is a stable identifier derived from the category + the
site's own attachment id, so Phase 3 (PostgreSQL change detection) can use it as
a natural dedup key without guessing.


In [ ]:
class DGCAPublication(BaseModel):
    """A single raw publication record extracted from DGCA 'What's New'."""

    source_id: str
    title: str
    category: str
    published_date: Optional[str] = None       # normalized ISO YYYY-MM-DD, if parseable
    published_date_raw: str = ""
    publication_url: str = BASE_URL
    pdf_url: Optional[str] = None
    file_size_bytes: Optional[int] = None
    file_type: Optional[str] = None
    scraped_at: str = ""


print("DGCAPublication model ready.")


## 7. HTTP Session & Locale Setup

`get_dgca_session()` creates a `requests.Session`, visits the homepage once
to pick up cookies, then replicates the site's own `langChange('en')` AJAX
call so that subsequent requests (including the server-rendered Tenders
HTML) come back in English. Locale failure is treated as non-fatal --
logged as a warning, since the AJAX-backed categories are unaffected by it
(they take `baseLocale` as an explicit query parameter already).


In [ ]:
def get_dgca_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    resp = session.get(BASE_URL, timeout=REQUEST_TIMEOUT_SECONDS)
    resp.raise_for_status()
    logger.info(f"Homepage loaded (HTTP {resp.status_code}), session cookies established.")

    try:
        locale_params = {
            "baseLocale": "en_US",
            "requestType": "ApplicationRH",
            "actionVal": "localeChange",
            "screenId": "10000001",
        }
        locale_resp = session.post(SCAN_URL, params=locale_params, timeout=REQUEST_TIMEOUT_SECONDS)
        locale_resp.raise_for_status()
        logger.info("Locale switched to English (en_US) for this session.")
    except requests.RequestException as exc:
        logger.warning(f"Locale switch failed ({exc.__class__.__name__}); Tenders may come back in Hindi.")

    return session


print("Session helper ready.")


## 8. Scraper — Regulation / Guidance / Announcement (JSON API)

Calls the AJAX endpoint identified during inspection and parses the parallel
`contents` / `startDate` / `attachId` arrays. Defensive by design: any
missing key, non-JSON response (e.g. an HTML error page, which is what this
endpoint actually returns for an unrecognized `classification` value -- see
Section 12), or length mismatch between the arrays is logged and handled
without raising, so one bad category can't take down the whole run.


In [ ]:
def fetch_ajax_category(session: requests.Session, classification: str) -> List[DGCAPublication]:
    params = {
        "baseLocale": "en_US",
        "requestType": "ApplicationRH",
        "actionVal": "viewAllServiceForAJAX",
        "screenId": "10000001",
        "classification": classification,
    }
    try:
        resp = session.post(SCAN_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        resp.raise_for_status()
        resp.encoding = "utf-8"  # the endpoint sends no charset; avoid mojibake
        data = resp.json()
    except requests.RequestException as exc:
        logger.error(f"[{classification}] Network/HTTP error while fetching: {exc.__class__.__name__}: {exc}")
        return []
    except ValueError as exc:  # json.JSONDecodeError subclasses ValueError
        logger.error(f"[{classification}] Response was not valid JSON (site may have changed): {exc}")
        return []

    contents = data.get("contents", [])
    dates = data.get("startDate", [])
    attach_ids = data.get("attachId", [])

    if not contents:
        logger.info(f"[{classification}] No publications returned (0 records).")
        return []

    lengths = {len(contents), len(dates), len(attach_ids)}
    if len(lengths) > 1:
        logger.warning(
            f"[{classification}] Parallel arrays have mismatched lengths "
            f"(contents={len(contents)}, startDate={len(dates)}, attachId={len(attach_ids)}); "
            "truncating to the shortest to stay safe."
        )
    n = min(lengths) if lengths else 0

    now = datetime.now(timezone.utc).isoformat()
    publications = []
    for i in range(n):
        attach_id = attach_ids[i]
        publications.append(
            DGCAPublication(
                source_id=make_source_id(classification, attach_id),
                title=contents[i].strip(),
                category=classification,
                published_date=normalize_date(dates[i]),
                published_date_raw=dates[i],
                publication_url=BASE_URL,
                pdf_url=build_pdf_url(attach_id),
                scraped_at=now,
            )
        )

    logger.info(f"[{classification}] {len(publications)} publication(s) parsed.")
    return publications


print("AJAX category scraper ready.")


## 9. Scraper — Tenders (server-rendered HTML)

Parses the `#tenders` fragment of the homepage with BeautifulSoup. Uses
`.select()`/`.find()` with `None` checks throughout rather than assuming
every expected element exists, and skips (with a warning) any block it can't
parse instead of crashing the whole scrape -- the resilience requirement
from the project brief.


In [ ]:
ATTACH_ID_RE = re.compile(r"pdfDownload\('([^']+)'")
DATE_SIZE_RE = re.compile(r"([0-9]{2}\.[0-9]{2}\.[0-9]{4})")


def fetch_tenders(session: requests.Session) -> List[DGCAPublication]:
    try:
        resp = session.get(BASE_URL, timeout=REQUEST_TIMEOUT_SECONDS)
        resp.raise_for_status()
    except requests.RequestException as exc:
        logger.error(f"[Tender] Network/HTTP error while fetching homepage: {exc.__class__.__name__}: {exc}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    container = soup.find(id="tenders")
    if container is None:
        logger.error("[Tender] Could not find #tenders section — page structure may have changed.")
        return []

    blocks = container.select(".new-window-pdf")
    if not blocks:
        logger.info("[Tender] No tender entries found (0 records).")
        return []

    now = datetime.now(timezone.utc).isoformat()
    publications: List[DGCAPublication] = []
    for block in blocks:
        heading_link = block.select_one(".sub-heading a")
        content_p = block.select_one(".sub-content")
        if heading_link is None or content_p is None:
            logger.warning("[Tender] Skipping a block missing expected .sub-heading/.sub-content.")
            continue

        title = heading_link.get_text(strip=True)
        onclick = heading_link.get("onclick", "") or ""
        match = ATTACH_ID_RE.search(onclick)
        if not match:
            logger.warning(f"[Tender] Skipping '{title[:50]}...' — could not extract attachId from onclick.")
            continue
        attach_id = match.group(1)

        content_text = content_p.get_text(" ", strip=True)
        date_match = DATE_SIZE_RE.search(content_text)
        date_raw = date_match.group(1) if date_match else ""
        if not date_match:
            logger.warning(f"[Tender] No date found for '{title[:50]}...'; leaving published_date unset.")

        publications.append(
            DGCAPublication(
                source_id=make_source_id("Tender", attach_id),
                title=title,
                category="Tender",
                published_date=normalize_date(date_raw) if date_raw else None,
                published_date_raw=date_raw,
                publication_url=BASE_URL,
                pdf_url=build_pdf_url(attach_id),
                scraped_at=now,
            )
        )

    logger.info(f"[Tender] {len(publications)} publication(s) parsed.")
    return publications


print("Tenders scraper ready.")


## 10. Helpers — Date Normalization, Source ID, PDF URL

Small, independently testable utility functions used by both scrapers above.
`normalize_date` never raises -- an unparseable date is logged and returns
`None` rather than breaking the record.


In [ ]:
_DATE_FORMATS = ("%d/%m/%Y", "%d.%m.%Y")


def normalize_date(raw_date: str) -> Optional[str]:
    cleaned = raw_date.strip()
    for fmt in _DATE_FORMATS:
        try:
            return datetime.strptime(cleaned, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    logger.warning(f"Could not parse date '{raw_date}' with known formats {_DATE_FORMATS}.")
    return None


def make_source_id(category: str, attach_id_encoded: str) -> str:
    decoded = unquote(attach_id_encoded)
    safe = re.sub(r"[^A-Za-z0-9]+", "", decoded)[:24]
    return f"{category.lower()}-{safe}"


def build_pdf_url(attach_id_encoded: str) -> str:
    # attach_id_encoded is already percent-encoded by the site (e.g. "...%3D%3D");
    # insert as-is to avoid double-encoding.
    return f"{UPLOAD_URL}?flag=iframeAttachView&attachId={attach_id_encoded}&baseLocale=en_US"


print("Helper functions ready.")


## 11. Orchestrator — `scrape_whats_new()`

Runs all four categories, isolating failures per category (one category's
exception is caught inside its own fetch function already; this level adds a
final safety net plus a summary of what succeeded/failed) so a single bad
category never aborts the whole run.


In [ ]:
@dataclass
class ScrapeRunResult:
    publications: List[DGCAPublication] = field(default_factory=list)
    category_counts: dict = field(default_factory=dict)
    failed_categories: List[str] = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


def scrape_whats_new(session: requests.Session) -> ScrapeRunResult:
    result = ScrapeRunResult()

    for category in AJAX_CATEGORIES:
        try:
            items = fetch_ajax_category(session, category)
            result.category_counts[category] = len(items)
            result.publications.extend(items)
        except Exception as exc:  # last-resort safety net; fetch_* already handles expected errors
            logger.error(f"[{category}] Unexpected error, skipping category: {exc.__class__.__name__}: {exc}")
            result.failed_categories.append(category)

    for category in HTML_CATEGORIES:
        try:
            items = fetch_tenders(session)
            result.category_counts[category] = len(items)
            result.publications.extend(items)
        except Exception as exc:
            logger.error(f"[{category}] Unexpected error, skipping category: {exc.__class__.__name__}: {exc}")
            result.failed_categories.append(category)

    logger.info(
        f"Scrape complete: {len(result.publications)} total publication(s), "
        f"by category: {result.category_counts}, failed: {result.failed_categories or 'none'}"
    )
    return result


print("Orchestrator ready.")


## 12. Error Handling & Resilience Tests

Two real (not mocked) failure scenarios, matching the project's testing
strategy:

1. **Unrecognized category** -- the live endpoint actually returns an HTML
   error page (not JSON) for an unknown `classification` value. This
   exercises the "site structure changed" / bad-response path for real.
2. **Website unreachable** -- a request to a non-routable IP address with a
   short timeout, simulating "DGCA website unavailable".

Both must return an empty list and log a clear error, without raising.


In [ ]:
_resilience_session = get_dgca_session()

# 1. Unrecognized classification -> endpoint returns HTML, not JSON.
bad_category_result = fetch_ajax_category(_resilience_session, "NotARealCategory")
assert bad_category_result == [], "Expected graceful empty result for an invalid category."
print(f"Invalid-category test passed. Result: {bad_category_result}")

# 2. Simulated unreachable host (non-routable IP, short timeout).
_unreachable_session = requests.Session()
_unreachable_session.headers.update({"User-Agent": USER_AGENT})
try:
    _unreachable_session.get("http://10.255.255.1/", timeout=2)
    network_test_passed = False
except requests.RequestException as exc:
    logger.error(f"[resilience-test] Simulated unreachable host failed as expected: {exc.__class__.__name__}")
    network_test_passed = True
print(f"Unreachable-host test passed. Exception raised and caught: {network_test_passed}")


## 13. End-to-End Test (Live Run)

Runs the real scrape against the live DGCA site using the session already
established in Section 12.


In [ ]:
run_result = scrape_whats_new(_resilience_session)
run_result


## 14. Save Raw Snapshot

Persists this run's results as a timestamped JSON file under `data/raw/` --
the input Phase 3's PostgreSQL change-detection layer will consume. No database
work happens in this notebook; this is just today's raw capture.


In [ ]:
def save_snapshot(result: ScrapeRunResult) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    path = RAW_DATA_DIR / f"whats_new_{timestamp}.json"
    payload = {
        "started_at": result.started_at,
        "category_counts": result.category_counts,
        "failed_categories": result.failed_categories,
        "publications": [p.model_dump() for p in result.publications],
    }
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


snapshot_path = save_snapshot(run_result)
logger.info(f"Snapshot saved to {snapshot_path}")
print(f"Snapshot saved to: {snapshot_path}")


## 15. Results

In [ ]:
print("=" * 70)
print("SCRAPE RUN RESULT")
print("=" * 70)
print(f"Started at        : {run_result.started_at}")
print(f"Total publications: {len(run_result.publications)}")
print(f"By category       : {run_result.category_counts}")
print(f"Failed categories : {run_result.failed_categories or 'none'}")
print("=" * 70)

for pub in run_result.publications[:10]:
    print(f"- [{pub.category}] {pub.title[:90]}")
    print(f"    published: {pub.published_date or '(unparsed) ' + pub.published_date_raw}")
    print(f"    pdf_url  : {pub.pdf_url}")
    print()

if len(run_result.publications) > 10:
    print(f"... and {len(run_result.publications) - 10} more (see {snapshot_path.name}).")


## 16. Next Steps

**Phase 2 status: scraper built and tested against the live DGCA site.**
Real publications were extracted from all reachable categories (see Section
15's counts) and saved to `data/raw/`.

### Known limitations to revisit

- If the Tenders locale-switch stops working (site change), Tenders titles
  would come back in Hindi rather than failing outright -- worth an explicit
  language check in a later hardening pass.
- No retry/backoff on transient network errors yet (Phase 1's
  `retry_with_backoff` pattern could be reused here if DGCA proves flaky).
- `attachMstVOListTwo` (file size/type) from the JSON API is not yet parsed
  into `file_size_bytes`/`file_type` -- straightforward to add when needed.

### Not started yet (by design)

- Phase 3: PostgreSQL change-detection storage (new vs. already-processed vs.
  updated), using `source_id` from this notebook as the natural key
- Phase 4: PDF download & text extraction
- Phase 5: Old-vs-new AI comparison
- Phase 6: AI summary integration into the scraped-data flow
- Phase 7: Agentic orchestration
- Phase 8: Streamlit dashboard

Waiting for confirmation this scraper's output looks right before starting
Phase 3.
